# Nanonis data readers: offline tutorial

This notebook demonstrates every implemented file reader against the committed files in `data_example/`:

- `read_sxm` for scan images;
- `read_3ds` for grid spectroscopy;
- `read_dat` for point spectroscopy;
- `read_session` for an explicit, separate session configuration snapshot.

It also demonstrates non-finite validation, immutable models, provenance, explicit transforms, xarray conversion, orientation declarations, and optional QCoDeS persistence. This notebook performs **no hardware I/O** and is safe to run from top to bottom. QCoDeS writes remain disabled unless explicitly enabled.


## 1. Import every public reader and analysis helper

Start Jupyter from the repository root or `examples/`. Install the project and visualization extras with `python -m pip install -e ".[xarray,qcodes]"`. Matplotlib is used only for the plots below.


In [ ]:
from __future__ import annotations

import logging
import math
import sys
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "nanonis").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "nanonis").exists():
    raise RuntimeError("Start Jupyter from the repository root or examples/")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from nanonis.data import (  # noqa: E402
    read_3ds,
    read_dat,
    read_session,
    read_sxm,
)
from nanonis.data.adapters import (  # noqa: E402
    dat_to_xarray,
    grid_3ds_to_xarray,
    orient_sxm_array,
    sxm_raw_orders,
    sxm_to_xarray,
)
from nanonis.data.transforms import (  # noqa: E402
    subtract_average,
    subtract_line,
    subtract_plane,
    subtract_vaverage,
    subtract_vline,
)
from nanonis.data.validation import inspect_non_finite  # noqa: E402
from nanonis.types import NaNPolicy  # noqa: E402

logging.basicConfig(level=logging.INFO)
print("Repository root:", ROOT)


## 2. Locate the real offline fixtures

These files are the byte-level regression oracles used by the test suite. No controller connection is involved.


In [ ]:
DATA = ROOT / "data_example"
SXM_PATH = DATA / "FGT_0030.sxm"
GRID_PATH = DATA / "Grid Spectroscopy023.3ds"
DAT_PATH = DATA / "Bias-Spectroscopy_00146.dat"
SESSION_PATH = DATA / "Nanonis-Session.ini"

fixture_paths = (SXM_PATH, GRID_PATH, DAT_PATH, SESSION_PATH)
for path in fixture_paths:
    if not path.is_file():
        raise FileNotFoundError(path)
    print(f"{path.name:36s} {path.stat().st_size / 1024:10.1f} KiB")


## 3. Read the SXM scan image

`read_sxm` preserves the file's raw matrix order. Each channel explicitly declares whether forward, backward, or both frames exist. Orientation is interpreted later by an adapter. The model owns immutable array copies, the complete frozen parser header, geometry in SI units, and compact provenance metadata.


In [ ]:
sxm = read_sxm(SXM_PATH, nan_policy=NaNPolicy.WARN)

print("Geometry:", sxm.region)
print("Pixels × lines:", sxm.pixels, "×", sxm.lines)
print("Raw SCAN_DIR:", sxm.scan_direction)
print("Parser:", sxm.parser)
for channel in sxm.channels:
    shapes = {direction: channel.data(direction).shape for direction in channel.directions}
    print(f"{channel.name:20s} unit={channel.unit!r} directions={channel.directions} shapes={shapes}")

print("Non-finite diagnostics:", inspect_non_finite(sxm))
print("First array is writable:", sxm.channels[0].data(sxm.channels[0].directions[0]).flags.writeable)
print("Compact provenance:")
pprint(sxm.to_metadata())


## 4. Convert SXM to an oriented xarray Dataset and display every frame

The confirmed SXM file convention is applied only here: `SCAN_DIR == "up"` flips rows, and backward frames flip columns. The resulting arrays use top-to-bottom, left-to-right physical X/Y coordinate grids. `sxm_raw_orders` exposes the pre-adapter interpretation explicitly.


In [ ]:
for channel in sxm.channels:
    for direction in channel.directions:
        print(channel.name, direction, "raw orders:", sxm_raw_orders(sxm.scan_direction, direction))

sxm_dataset = sxm_to_xarray(sxm)
display(sxm_dataset)

variables = list(sxm_dataset.data_vars)
ncols = 2
nrows = math.ceil(len(variables) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows), squeeze=False)
for axis, variable in zip(axes.flat, variables):
    image = axis.imshow(sxm_dataset[variable].values, origin="upper", cmap="viridis")
    axis.set_title(f"{variable} [{sxm_dataset[variable].attrs['units']}]")
    axis.set_xlabel("column")
    axis.set_ylabel("row")
    fig.colorbar(image, ax=axis, shrink=0.8)
for axis in axes.flat[len(variables):]:
    axis.set_visible(False)
fig.suptitle("All SXM channel/direction frames (physically oriented)")
fig.tight_layout()
plt.show()


## 5. Apply every opt-in image transform

Readers never flatten data. These pure functions return new arrays and fit finite values only. The raw immutable domain data remains unchanged. This cell applies the complete correction catalogue to one physically oriented frame.


In [ ]:
source_channel = sxm.channels[0]
source_direction = source_channel.directions[0]
raw_frame = source_channel.data(source_direction)
oriented_frame = orient_sxm_array(
    raw_frame,
    scan_direction=sxm.scan_direction,
    data_direction=source_direction,
)
transformed = {
    "raw": oriented_frame,
    "subtract_average": subtract_average(oriented_frame),
    "subtract_vaverage": subtract_vaverage(oriented_frame),
    "subtract_line": subtract_line(oriented_frame),
    "subtract_vline": subtract_vline(oriented_frame),
    "subtract_plane": subtract_plane(oriented_frame),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 9), squeeze=False)
for axis, (name, values) in zip(axes.flat, transformed.items()):
    image = axis.imshow(values, origin="upper", cmap="viridis")
    axis.set_title(name)
    fig.colorbar(image, ax=axis, shrink=0.75)
fig.suptitle(f"Transforms: {source_channel.name} [{source_direction}]")
fig.tight_layout()
plt.show()
assert not np.shares_memory(raw_frame, transformed["subtract_plane"])


## 6. Read the 3DS grid-spectroscopy file

`read_3ds` returns one 3-D cube per channel, a sweep axis, per-pixel fixed/experimental parameter maps, and frame geometry. This fixture's sweep is decreasing. One fixed-parameter map contains NaNs, so `NaNPolicy.ALLOW` makes that choice explicit for the demonstration.


In [ ]:
grid = read_3ds(GRID_PATH, nan_policy=NaNPolicy.ALLOW)

print("Geometry:", grid.region)
print("Pixels × lines:", grid.pixels, "×", grid.lines)
print("Sweep:", grid.sweep_signal.name, grid.sweep_signal.unit, grid.sweep_signal.direction)
print("Sweep endpoints:", grid.sweep_signal.values[[0, -1]])
for channel in grid.channels:
    print(f"{channel.name:20s} unit={channel.unit!r} cube={channel.values.shape}")
print("Parameter maps:", tuple(grid.fixed_parameters))
print("Non-finite diagnostics:", inspect_non_finite(grid))
pprint(grid.to_metadata())


## 7. Convert 3DS to xarray with an explicit orientation declaration

This rig has no paired in-memory acquisition oracle for 3DS orientation. The adapter therefore has no silent row-order default and does not flip the cube. Set `GRID_ROW_ORDER` only after checking your controller/file convention; the selected interpretation is recorded as unverified metadata.


In [ ]:
GRID_ROW_ORDER = "bottom_to_top"  # Explicit assumption; not ground-truth validated.
GRID_COLUMN_ORDER = "left_to_right"
grid_dataset = grid_3ds_to_xarray(
    grid,
    row_order=GRID_ROW_ORDER,
    column_order=GRID_COLUMN_ORDER,
)
display(grid_dataset)
print("Orientation metadata:", {
    key: grid_dataset.attrs[key]
    for key in ("raw_row_order", "raw_column_order", "orientation_ground_truth_validated")
})

channel = grid.channels[0]
parameter_name, parameter_map = next(iter(grid.fixed_parameters.items()))
mid_row, mid_column = grid.lines // 2, grid.pixels // 2
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(parameter_map, origin="lower", cmap="viridis")
axes[0].set_title(f"Parameter map: {parameter_name}")
axes[1].imshow(channel.values[:, :, channel.values.shape[-1] // 2], origin="lower", cmap="coolwarm")
axes[1].set_title(f"{channel.name}: middle sweep plane")
axes[2].plot(grid.sweep_signal.values, channel.values[mid_row, mid_column, :])
axes[2].set_xlabel(f"{grid.sweep_signal.name} ({grid.sweep_signal.unit})")
axes[2].set_ylabel(f"{channel.name} ({channel.unit})")
axes[2].set_title("Spectrum at the grid centre")
fig.tight_layout()
plt.show()


## 8. Read the DAT bias-spectroscopy file

`read_dat` returns immutable typed columns. The fixture includes filtered columns with intentional NaNs, so this cell uses `NaNPolicy.ALLOW`. Only this bias-spectroscopy DAT variant is currently fixture-backed; single-spectrum and history variants require their own real regression fixtures.


In [ ]:
dat = read_dat(DAT_PATH, nan_policy=NaNPolicy.ALLOW)

print("Points:", dat.points)
for column in dat.columns:
    finite = int(np.isfinite(column.values).sum())
    print(f"{column.name:24s} unit={column.unit!r} shape={column.values.shape} finite={finite}")
print("Non-finite diagnostics:", inspect_non_finite(dat))
pprint(dat.to_metadata())


## 9. Convert DAT to xarray and plot representative columns

The xarray adapter preserves the file's sample order and exposes each column with its SI unit and original label.


In [ ]:
dat_dataset = dat_to_xarray(dat)
display(dat_dataset)

x_name = "bias_calc"
y_names = [name for name in ("current", "li_demod_1_x", "li_demod_1_y") if name in dat_dataset]
fig, axes = plt.subplots(len(y_names), 1, figsize=(8, 3.5 * len(y_names)), squeeze=False)
for axis, y_name in zip(axes.flat, y_names):
    axis.plot(dat_dataset[x_name], dat_dataset[y_name])
    axis.set_xlabel(f"{dat_dataset[x_name].attrs['long_name']} ({dat_dataset[x_name].attrs['units']})")
    axis.set_ylabel(f"{dat_dataset[y_name].attrs['long_name']} ({dat_dataset[y_name].attrs['units']})")
    axis.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 10. Read the optional session configuration explicitly

Measurement files remain self-describing and never auto-load this mutable session snapshot. `read_session` is a separate opt-in operation. It retains every section and provides typed convenience access for commonly useful modules.


In [ ]:
session = read_session(SESSION_PATH)

print("Section count:", len(session.modules))
print("Sections:", tuple(session.modules))
print("Lock-in module:", session.lock_in.name if session.lock_in else None)
print("Atom tracking module:", session.atom_tracking.name if session.atom_tracking else None)
print("Bias spectroscopy module:", session.bias_spectroscopy.name if session.bias_spectroscopy else None)
if session.bias_spectroscopy:
    print("Bias spectroscopy settings preview:")
    pprint(dict(list(session.bias_spectroscopy.values.items())[:12]))
pprint(session.to_metadata())


## 11. Inspect full headers only when needed

Normal metadata and QCoDeS persistence use compact header summaries. The complete frozen parser header remains on each model and can be converted explicitly. Keep this disabled unless the full payload is needed because SXM headers can be large.


In [ ]:
INCLUDE_FULL_HEADERS = False
if INCLUDE_FULL_HEADERS:
    full_sxm_metadata = sxm.to_metadata(include_header=True)
    print("Full SXM header fields:", len(full_sxm_metadata["header"]))
    pprint(full_sxm_metadata["header"])
else:
    print("Full-header conversion skipped; use model.header for direct read-only access.")


## 12. Optional QCoDeS persistence for every measurement model

This is acquire-first persistence: it writes the completed domain objects and never controls hardware. The safe default is `False`. Set it to `True` only when you want to create/update the SQLite database shown below. Session configuration is context, not measurement data, and is therefore not persisted through these adapters.


In [ ]:
SAVE_TO_QCODES = False
QCODES_DB = ROOT / "data_readers_demo.db"

if SAVE_TO_QCODES:
    from qcodes.dataset import initialise_or_create_database_at, load_or_create_experiment
    from nanonis.qcodes.data import (
        add_dat_data,
        add_grid_3ds_data,
        add_sxm_data,
        create_dat_measurement,
        create_grid_3ds_measurement,
        create_sxm_measurement,
    )

    initialise_or_create_database_at(QCODES_DB)
    experiment = load_or_create_experiment(
        experiment_name="Nanonis file readers demo",
        sample_name="offline fixtures",
    )

    sxm_measurement, sxm_registration = create_sxm_measurement(experiment, sxm)
    with sxm_measurement.run() as datasaver:
        add_sxm_data(datasaver, sxm_registration, sxm)
        print("SXM run id:", datasaver.run_id)

    grid_measurement, grid_registration = create_grid_3ds_measurement(experiment, grid)
    with grid_measurement.run() as datasaver:
        add_grid_3ds_data(datasaver, grid_registration, grid)
        print("3DS run id:", datasaver.run_id)

    dat_measurement, dat_registration = create_dat_measurement(experiment, dat)
    with dat_measurement.run() as datasaver:
        add_dat_data(datasaver, dat_registration, dat)
        print("DAT run id:", datasaver.run_id)

    print("Database:", QCODES_DB)
else:
    print("QCoDeS write skipped. Set SAVE_TO_QCODES=True to persist all three measurement models.")


## 13. Summary

The reader boundary is now visible end to end:

1. Nanonis files are parsed into immutable SI-domain models.
2. Structural validation happens before return; `NaNPolicy` makes partial data handling explicit.
3. Raw matrix order and raw values remain available.
4. Orientation and transforms are separate, opt-in analysis operations.
5. xarray and QCoDeS consume completed domain data without touching hardware.
6. Session configuration is loaded explicitly and never treated as authoritative per-measurement provenance.
